In [1]:
# ============================================================
# Cell 1 — Mount Drive, imports, globals
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy import stats as scipy_stats
from scipy.special import exp1

from sklearn.ensemble import BaggingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — training will be slow. Runtime > Change runtime type > GPU")

DRIVE_BASE = Path("/content/drive/MyDrive/DSP Project")
AUDIO_DIR = DRIVE_BASE / "esc50_data/ESC-50-master/audio"
META_PATH = DRIVE_BASE / "esc50_data/ESC-50-master/meta/esc50.csv"
TRAIN_CSV = DRIVE_BASE / "train_mfcc.csv"
TEST_CSV = DRIVE_BASE / "test_mfcc.csv"

REAL_NOISE_PATH = str(DRIVE_BASE / "traffic_noise.wav")

OUT_DIR = DRIVE_BASE / "mfcc_unet_denoiser_multirun"
OUT_MODELS = OUT_DIR / "models"
OUT_RESULTS = OUT_DIR / "results"
for _d in [OUT_MODELS, OUT_RESULTS]:
    _d.mkdir(parents=True, exist_ok=True)

SR = 44_100
CLIP_LEN = SR * 5  # 220500 samples
N_MFCC = 13
N_FFT_MFCC = 2048
HOP_MFCC = 1024
N_MELS = 128
FMIN = 50
FMAX = 8_000
TARGET_RMS = 0.1

# STFT settings for U-Net denoiser
N_FFT_UNET = 1024
HOP_UNET = 256
WIN_UNET = 1024

# ESC-50 fold protocol
TRAIN_FOLDS = [1, 2, 3, 4]
TEST_FOLD = 5

# U-Net training hyperparameters
UNET_EPOCHS = 10
UNET_LR = 1e-3
UNET_BATCH_SIZE = 8
UNET_WEIGHT_DECAY = 1e-5
EARLY_STOP_PATIENCE = 4

# U-Net evaluation batching
UNET_INFER_BATCH = 16

# Multi-run settings
N_RUNS = 5
BASE_SEED = 42

# SNR grid
SNR_LEVELS = [20, 15, 10, 5, 0, -5, -10]

# Noise generators to evaluate
NOISE_GENERATORS = ["traffic", "gaussian"]  # white noise excluded — paper scope is traffic + Gaussian only
NOISE_TAG = "-".join(NOISE_GENERATORS)  # used to namespace cached checkpoints by noise config

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(BASE_SEED)
print(f"Output folder : {OUT_DIR}")
print(f"N_RUNS : {N_RUNS}")
print(f"SNR levels : {SNR_LEVELS}")
print(f"UNET_INFER_BATCH : {UNET_INFER_BATCH}")


Mounted at /content/drive
Device : cuda
GPU : Tesla T4
VRAM : 15.6 GB
Output folder : /content/drive/MyDrive/DSP Project/mfcc_unet_denoiser_multirun
N_RUNS : 5
SNR levels : [20, 15, 10, 5, 0, -5, -10]
UNET_INFER_BATCH : 16


In [2]:
# ============================================================
# Cell 2 — Preprocessing helpers
# ============================================================

def pad_or_trim(y: np.ndarray) -> np.ndarray:
    if len(y) > CLIP_LEN:
        return y[:CLIP_LEN]
    return np.pad(y, (0, CLIP_LEN - len(y))).astype(np.float32)

def rms_normalize(y: np.ndarray, target: float = TARGET_RMS) -> np.ndarray:
    rms = np.sqrt(np.mean(y ** 2))
    if rms < 1e-9:
        return y.astype(np.float32)
    return (y * (target / rms)).astype(np.float32)

SILENCE_THRESHOLD = 0.30

def silence_trim(y: np.ndarray, top_db: int = 20) -> np.ndarray:
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    silence_frac = 1.0 - len(y_trimmed) / max(len(y), 1)
    if silence_frac >= SILENCE_THRESHOLD and len(y_trimmed) >= 512:
        return pad_or_trim(y_trimmed)
    return y

def pre_emphasis(y: np.ndarray, coeff: float = 0.97) -> np.ndarray:
    return np.append(y[0], y[1:] - coeff * y[:-1]).astype(np.float32)

def preprocess(y: np.ndarray) -> np.ndarray:
    y = rms_normalize(pad_or_trim(y))
    y = silence_trim(y)
    y = pre_emphasis(y)
    y = rms_normalize(y)
    return y

def load_audio_clip(file_path: str) -> np.ndarray:
    y, _ = librosa.load(file_path, sr=SR, mono=True)
    target_len = SR * 5
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    else:
        y = y[:target_len]
    return rms_normalize(y)

print("Preprocessing helpers ready.")


Preprocessing helpers ready.


In [3]:
# ============================================================
# Cell 3 — MFCC feature extraction
# ============================================================

def compute_stats(matrix: np.ndarray) -> np.ndarray:
    return np.concatenate([
        np.mean(matrix, axis=1),
        np.std(matrix, axis=1),
        np.max(matrix, axis=1),
        np.min(matrix, axis=1),
        np.median(matrix, axis=1),
    ])

def extract_features(y: np.ndarray, sr: int = SR) -> np.ndarray:
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=N_MFCC,
        n_fft=N_FFT_MFCC, hop_length=HOP_MFCC,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    )
    delta1 = librosa.feature.delta(mfcc, order=1)
    delta2 = librosa.feature.delta(mfcc, order=2)
    return np.concatenate([
        compute_stats(mfcc),
        compute_stats(delta1),
        compute_stats(delta2),
    ]).astype(np.float32)

FEAT_COLS = [
    f"{coeff}{idx}{stat}"
    for coeff in ["mfcc", "delta", "delta2"]
    for idx in range(1, N_MFCC + 1)
    for stat in ["mean", "std", "max", "min", "med"]
]
assert len(FEAT_COLS) == 195, "Feature count mismatch!"
print(f"Feature extractor: {len(FEAT_COLS)}-d vector per clip.")


Feature extractor: 195-d vector per clip.


In [4]:
# ============================================================
# Cell 4 — Load MFCC CSVs & train ensemble
# ============================================================

print("Loading MFCC CSVs …")
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

FEATURE_COLS = [c for c in train_df.columns if c not in ("label", "fold")]
X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = train_df["label"].values
X_test = test_df[FEATURE_COLS].values.astype(np.float32)
y_test = test_df["label"].values

CLASSES = sorted(np.unique(y_train))
label2idx = {c: i for i, c in enumerate(CLASSES)}

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Subspace Discriminant Ensemble …")
ensemble_model = BaggingClassifier(
    estimator=LinearDiscriminantAnalysis(),
    n_estimators=30,
    max_features=98,
    max_samples=1.0,
    bootstrap=False,
    bootstrap_features=True,
    random_state=BASE_SEED,
    n_jobs=-1,
)
ensemble_model.fit(X_train_scaled, y_train)

clean_preds = ensemble_model.predict(X_test_scaled)
CLEAN_ACC = accuracy_score(y_test, clean_preds) * 100
print(f"\nClean baseline accuracy : {CLEAN_ACC:.2f}%")
print("MATLAB target : 53.0%")


Loading MFCC CSVs …
Training Subspace Discriminant Ensemble …

Clean baseline accuracy : 55.75%
MATLAB target : 53.0%


In [5]:
# ============================================================
# Cell 5 — Load ESC-50 test audio
# ============================================================

meta_df = pd.read_csv(META_PATH)
test_meta = meta_df[meta_df["fold"] == TEST_FOLD].reset_index(drop=True)
print(f"Test fold {TEST_FOLD}: {len(test_meta)} clips")

test_waves = []
test_fnames = []
test_labels_audio = []

for _, row in tqdm(test_meta.iterrows(), total=len(test_meta), desc="Loading audio"):
    fpath = str(AUDIO_DIR / row["filename"])
    try:
        y = load_audio_clip(fpath)
        test_waves.append(y)
        test_fnames.append(row["filename"])
        test_labels_audio.append(row["category"])
    except Exception as err:
        print(f" Error: {row['filename']}: {err}")

test_waves = np.array(test_waves, dtype=np.float32)
test_labels_audio = np.array(test_labels_audio)
print(f"Loaded {len(test_waves)} clips | Shape: {test_waves.shape}")


Test fold 5: 400 clips


Loading audio: 100%|██████████| 400/400 [03:10<00:00,  2.09it/s]

Loaded 400 clips | Shape: (400, 220500)


In [6]:
# ============================================================
# Cell 6 — Load traffic noise & noise functions
# ============================================================

if os.path.exists(REAL_NOISE_PATH):
    print(f"Found traffic noise: {REAL_NOISE_PATH}")
else:
    print(f"Traffic noise not found at: {REAL_NOISE_PATH}")
    print("Please upload traffic_noise.wav now …")
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.makedirs(os.path.dirname(REAL_NOISE_PATH), exist_ok=True)
    shutil.copy(fname, REAL_NOISE_PATH)
    print(f"Saved → {REAL_NOISE_PATH}")

_real_noise_wav, _sr_in = torchaudio.load(REAL_NOISE_PATH)
if _sr_in != SR:
    _real_noise_wav = torchaudio.functional.resample(_real_noise_wav, _sr_in, SR)
if _real_noise_wav.shape[0] > 1:
    _real_noise_wav = _real_noise_wav.mean(dim=0, keepdim=True)
_raw_noise = _real_noise_wav.squeeze(0).numpy().astype(np.float32)
print(f"Duration : {len(_raw_noise)/SR:.1f}s | Samples: {len(_raw_noise)}")

def _fit_noise_to_length(noise: np.ndarray, length: int, rng) -> np.ndarray:
    if len(noise) < length:
        reps = int(np.ceil(length / len(noise)))
        noise = np.tile(noise, reps)
    if len(noise) > length:
        max_start = len(noise) - length
        start = int(rng.integers(0, max_start + 1))
        noise = noise[start:start + length]
    return noise

def add_traffic_noise(wav_np: np.ndarray, target_snr_db: float, rng=None) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    noise = _fit_noise_to_length(_raw_noise, len(wav_np), rng)
    p_sig = np.mean(wav_np ** 2)
    p_n = p_sig / (10 ** (target_snr_db / 10.0))
    noise = noise * np.sqrt(p_n / (np.mean(noise ** 2) + 1e-10))
    return np.clip(wav_np + noise, -1.0, 1.0).astype(np.float32)

def add_gaussian_noise(wav_np: np.ndarray, target_snr_db: float, rng=None) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    p_sig = np.mean(wav_np ** 2)
    p_n = p_sig / (10 ** (target_snr_db / 10.0))
    noise = rng.standard_normal(len(wav_np)).astype(np.float32) * np.sqrt(p_n)
    return np.clip(wav_np + noise, -1.0, 1.0).astype(np.float32)

NOISE_FN_MAP = {
    "traffic": add_traffic_noise,
    "gaussian": add_gaussian_noise,
}

def measure_snr_input(clean, noisy):
    noise = noisy - clean
    p_sig = np.mean(clean ** 2)
    p_noise = np.mean(noise ** 2)
    return 10 * np.log10(p_sig / (p_noise + 1e-10))

def measure_snr_output(clean, denoised):
    noise = denoised - clean
    p_sig = np.mean(clean ** 2)
    p_noise = np.mean(noise ** 2)
    return 10 * np.log10(p_sig / (p_noise + 1e-10))


Found traffic noise: /content/drive/MyDrive/DSP Project/traffic_noise.wav
Duration : 5.1s | Samples: 223232


In [7]:
# ============================================================
# Cell 7 — STFT helpers
# ============================================================

def stft_magnitude_phase(wav_np: np.ndarray):
    t = torch.tensor(wav_np, dtype=torch.float32).unsqueeze(0)
    window = torch.hann_window(WIN_UNET)
    S = torch.stft(
        t, n_fft=N_FFT_UNET, hop_length=HOP_UNET,
        win_length=WIN_UNET, window=window,
        return_complex=True,
    )
    mag = S.abs()
    phase = torch.angle(S)
    return mag, phase

def istft_from_mask(mag: torch.Tensor, phase: torch.Tensor, orig_len: int) -> np.ndarray:
    S_hat = mag * torch.exp(1j * phase)
    window = torch.hann_window(WIN_UNET)
    wav = torch.istft(
        S_hat, n_fft=N_FFT_UNET, hop_length=HOP_UNET,
        win_length=WIN_UNET, window=window,
        length=orig_len,
    )
    return wav.squeeze(0).numpy()

print("STFT helpers ready.")


STFT helpers ready.


In [8]:
# ============================================================
# Cell 8 — U-Net architecture
# ============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNetDenoiser(nn.Module):
    def __init__(self, base_ch: int = 32):
        super().__init__()
        self.enc1 = DoubleConv(1,         base_ch)
        self.enc2 = DoubleConv(base_ch,   base_ch * 2)
        self.enc3 = DoubleConv(base_ch*2, base_ch * 4)
        self.pool = nn.MaxPool2d(2)

        self.bot  = DoubleConv(base_ch*4, base_ch * 8)

        self.up3  = nn.ConvTranspose2d(base_ch*8, base_ch*4, 2, stride=2)
        self.dec3 = DoubleConv(base_ch*8, base_ch*4)
        self.up2  = nn.ConvTranspose2d(base_ch*4, base_ch*2, 2, stride=2)
        self.dec2 = DoubleConv(base_ch*4, base_ch*2)
        self.up1  = nn.ConvTranspose2d(base_ch*2, base_ch,   2, stride=2)
        self.dec1 = DoubleConv(base_ch*2, base_ch)

        self.head = nn.Sequential(
            nn.Conv2d(base_ch, 1, 1),
            nn.Sigmoid(),
        )

    def _pad_to_match(self, x, ref):
        dh = ref.shape[2] - x.shape[2]
        dw = ref.shape[3] - x.shape[3]
        if dh != 0 or dw != 0:
            x = F.pad(x, [0, dw, 0, dh])
        return x

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self._pad_to_match(self.up3(b),  e3), e3], dim=1))
        d2 = self.dec2(torch.cat([self._pad_to_match(self.up2(d3), e2), e2], dim=1))
        d1 = self.dec1(torch.cat([self._pad_to_match(self.up1(d2), e1), e1], dim=1))
        return self.head(d1)

_dummy = torch.zeros(1, 1, 513, 100)
_unet = UNetDenoiser()
_out = _unet(_dummy)
print(f"U-Net input : {_dummy.shape}")
print(f"U-Net output : {_out.shape} (should match input spatial dims)")
del _dummy, _unet, _out


U-Net input : torch.Size([1, 1, 513, 100])
U-Net output : torch.Size([1, 1, 513, 100]) (should match input spatial dims)


In [9]:
# ============================================================
# Cell 9 — Dataset for U-Net training
# ============================================================

class STFTDenoiserDataset(Dataset):
    """Generates (noisy_mag, clean_mag) pairs on-the-fly at random SNR."""
    def __init__(self, meta_df, audio_dir, noise_fn_map,
                 snr_range=(-20, 20), seed=0):
        self.meta      = meta_df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.noise_fns = list(noise_fn_map.values())
        self.snr_min, self.snr_max = snr_range
        self.rng       = np.random.default_rng(seed)

    def __len__(self):
        return len(self.meta)

    def _load(self, filename):
        y = load_audio_clip(str(self.audio_dir / filename))
        return y

    def __getitem__(self, idx):
        y_clean = self._load(self.meta.loc[idx, "filename"])
        snr     = float(self.rng.uniform(self.snr_min, self.snr_max))
        noise_fn = self.noise_fns[int(self.rng.integers(len(self.noise_fns)))]
        try:
            y_noisy = noise_fn(y_clean, snr, rng=self.rng)
        except TypeError:
            y_noisy = noise_fn(y_clean, snr)

        window = torch.hann_window(WIN_UNET)

        def _mag(wav):
            t = torch.tensor(wav, dtype=torch.float32).unsqueeze(0)
            S = torch.stft(
                t, n_fft=N_FFT_UNET, hop_length=HOP_UNET,
                win_length=WIN_UNET, window=window,
                return_complex=True
            )
            return S.abs()

        return _mag(y_noisy), _mag(y_clean)

all_meta = pd.read_csv(META_PATH)
train_meta = all_meta[all_meta["fold"].isin(TRAIN_FOLDS)].reset_index(drop=True)

_tr, _va = train_test_split(
    train_meta, test_size=0.10,
    stratify=train_meta["category"],
    random_state=BASE_SEED
)
train_meta_inner = _tr.reset_index(drop=True)
val_meta_inner = _va.reset_index(drop=True)

print(f"U-Net train clips : {len(train_meta_inner)}")
print(f"U-Net val clips : {len(val_meta_inner)}")


U-Net train clips : 1440
U-Net val clips : 160


In [10]:
# ============================================================
# Cell 10 — train_unet function
# ============================================================

from tqdm.auto import tqdm  # nicer in-notebook progress bars (overrides plain tqdm import from Cell 1)


def train_unet(seed, save_path=None):
    set_seed(seed)

    ds_train = STFTDenoiserDataset(
        train_meta_inner, AUDIO_DIR, NOISE_FN_MAP,
        snr_range=(-20, 20), seed=seed,
    )
    ds_val = STFTDenoiserDataset(
        val_meta_inner, AUDIO_DIR, NOISE_FN_MAP,
        snr_range=(-20, 20), seed=seed + 10000,
    )

    dl_train = DataLoader(
        ds_train, batch_size=UNET_BATCH_SIZE,
        shuffle=True, num_workers=2,
        pin_memory=(device.type == "cuda")
    )
    dl_val = DataLoader(
        ds_val, batch_size=UNET_BATCH_SIZE,
        shuffle=False, num_workers=2,
        pin_memory=(device.type == "cuda")
    )

    unet = UNetDenoiser().to(device)
    optimiser = torch.optim.Adam(
        unet.parameters(),
        lr=UNET_LR, weight_decay=UNET_WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, patience=2, factor=0.5
    )
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    patience_ctr = 0

    print("\nStarting U‑Net training...\n")

    for epoch in range(1, UNET_EPOCHS + 1):

        # -----------------------------
        # TRAINING
        # -----------------------------
        unet.train()
        tr_loss = 0.0

        pbar = tqdm(dl_train, desc=f"Epoch {epoch}/{UNET_EPOCHS} — Training", leave=False)
        for noisy_mag, clean_mag in pbar:
            noisy_mag = noisy_mag.to(device)
            clean_mag = clean_mag.to(device)

            max_val = noisy_mag.amax(dim=(2,3), keepdim=True).clamp(min=1e-8)
            noisy_norm = noisy_mag / max_val
            clean_norm = clean_mag / max_val
            mask_target = (clean_norm / noisy_norm.clamp(min=1e-8)).clamp(0, 1)

            pred_mask = unet(noisy_norm)
            loss = criterion(pred_mask, mask_target)

            optimiser.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimiser.step()

            tr_loss += loss.item()
            pbar.set_postfix({"batch_loss": loss.item()})

        tr_loss /= len(dl_train)

        # -----------------------------
        # VALIDATION
        # -----------------------------
        unet.eval()
        va_loss = 0.0

        pbar_val = tqdm(dl_val, desc=f"Epoch {epoch}/{UNET_EPOCHS} — Validation", leave=False)
        with torch.no_grad():
            for noisy_mag, clean_mag in pbar_val:
                noisy_mag = noisy_mag.to(device)
                clean_mag = clean_mag.to(device)

                max_val = noisy_mag.amax(dim=(2,3), keepdim=True).clamp(min=1e-8)
                noisy_norm = noisy_mag / max_val
                clean_norm = clean_mag / max_val
                mask_target = (clean_norm / noisy_norm.clamp(min=1e-8)).clamp(0, 1)

                pred_mask = unet(noisy_norm)
                loss = criterion(pred_mask, mask_target)
                va_loss += loss.item()

                pbar_val.set_postfix({"val_loss": loss.item()})

        va_loss /= len(dl_val)
        scheduler.step(va_loss)

        # -----------------------------
        # EPOCH SUMMARY
        # -----------------------------
        current_lr = optimiser.param_groups[0]["lr"]
        improved = va_loss < best_val_loss

        print(
            f"Epoch {epoch:02d} | "
            f"Train={tr_loss:.5f} | "
            f"Val={va_loss:.5f} | "
            f"LR={current_lr:.2e} | "
            f"{'BEST ✔️' if improved else 'patience ' + str(patience_ctr+1)}"
        )

        if improved:
            best_val_loss = va_loss
            best_state = {k: v.cpu().clone() for k, v in unet.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

    unet.load_state_dict(best_state)
    if save_path:
        torch.save(best_state, save_path)
        print(f"\nSaved best model → {save_path}")

    return unet


In [11]:
# ============================================================
# Cell 11 — U-Net inference helpers
# ============================================================

def unet_denoise(unet: UNetDenoiser, wav_np: np.ndarray) -> np.ndarray:
    unet.eval()
    with torch.no_grad():
        window = torch.hann_window(WIN_UNET)
        t = torch.tensor(wav_np, dtype=torch.float32).unsqueeze(0)
        S = torch.stft(
            t, n_fft=N_FFT_UNET, hop_length=HOP_UNET,
            win_length=WIN_UNET, window=window,
            return_complex=True
        )
        mag = S.abs().unsqueeze(0).to(device)  # (1,1,F,T)
        max_val = mag.amax(dim=(2,3), keepdim=True).clamp(min=1e-8)
        mag_norm = mag / max_val
        mask = unet(mag_norm)
        masked_mag = (mask * mag_norm) * max_val
        masked_mag = masked_mag.squeeze(0).cpu()
        phase = torch.angle(S)
        S_hat = masked_mag * torch.exp(1j * phase)
        wav_out = torch.istft(
            S_hat, n_fft=N_FFT_UNET, hop_length=HOP_UNET,
            win_length=WIN_UNET, window=window,
            length=len(wav_np),
        )
    return wav_out.squeeze(0).numpy().astype(np.float32)

def unet_denoise_batch(unet: UNetDenoiser, wav_batch_np: np.ndarray) -> np.ndarray:
    unet.eval()
    with torch.no_grad():
        window = torch.hann_window(WIN_UNET)
        t = torch.tensor(wav_batch_np, dtype=torch.float32)  # (B, L)
        S = torch.stft(
            t, n_fft=N_FFT_UNET, hop_length=HOP_UNET,
            win_length=WIN_UNET, window=window,
            return_complex=True
        )  # (B, F, T)
        mag      = S.abs().unsqueeze(1).to(device)  # (B,1,F,T)
        max_val  = mag.amax(dim=(2, 3), keepdim=True).clamp(min=1e-8)
        mag_norm = mag / max_val
        mask     = unet(mag_norm)
        masked_mag = (mask * mag_norm) * max_val
        masked_mag = masked_mag.squeeze(1).cpu()  # (B,F,T)
        phase      = torch.angle(S)
        S_hat      = masked_mag * torch.exp(1j * phase)
        wav_out    = torch.istft(
            S_hat, n_fft=N_FFT_UNET, hop_length=HOP_UNET,
            win_length=WIN_UNET, window=window,
            length=wav_batch_np.shape[1],
        )
    return wav_out.numpy().astype(np.float32)

def infer_mfcc(wav_np: np.ndarray) -> str:
    feats = extract_features(wav_np, sr=SR)
    feats = scaler.transform(feats[np.newaxis, :])[0]
    return ensemble_model.predict(feats[np.newaxis, :])[0]

print("unet_denoise(), unet_denoise_batch(), and infer_mfcc() defined.")


unet_denoise(), unet_denoise_batch(), and infer_mfcc() defined.


In [12]:
# ============================================================
# Cell 12 — Multi-run experiment
# ============================================================

multirun_records = []
clean_run_acc = []

for run_idx in range(N_RUNS):
    run_seed = BASE_SEED + run_idx * 100
    print(f"\n{'='*70}")
    print(f" RUN {run_idx+1}/{N_RUNS} (seed={run_seed})")
    print(f"{'='*70}")
    set_seed(run_seed)

    ckpt_path = OUT_MODELS / f"unet_run{run_idx+1}_seed{run_seed}_{NOISE_TAG}.pth"  # noise-config-aware filename
    if ckpt_path.exists():
        print(f"  Loading cached checkpoint: {ckpt_path}")
        unet = UNetDenoiser().to(device)
        unet.load_state_dict(torch.load(ckpt_path, map_location=device))
    else:
        print("  Training U-Net …")
        unet = train_unet(seed=run_seed, save_path=ckpt_path)
    unet.eval()

    run_clean_preds = ensemble_model.predict(X_test_scaled)
    run_clean_acc   = accuracy_score(y_test, run_clean_preds) * 100
    clean_run_acc.append(run_clean_acc)
    print(f"  Clean accuracy this run: {run_clean_acc:.2f}%")

    run_rng = np.random.default_rng(run_seed + 999)

    for cond in NOISE_GENERATORS:
        noise_fn = NOISE_FN_MAP[cond]
        print(f"\n  Condition: {cond.upper()}")

        for snr in SNR_LEVELS:
            noisy_list = []
            for wav_np in test_waves:
                try:
                    noisy = noise_fn(wav_np, snr, rng=run_rng)
                except TypeError:
                    noisy = noise_fn(wav_np, snr)
                noisy_list.append(noisy)
            noisy_arr = np.stack(noisy_list).astype(np.float32)

            denoised_chunks = []
            for b0 in range(0, len(noisy_arr), UNET_INFER_BATCH):
                chunk = noisy_arr[b0:b0 + UNET_INFER_BATCH]
                denoised_chunks.append(unet_denoise_batch(unet, chunk))
            denoised_arr = np.concatenate(denoised_chunks, axis=0)

            noisy_preds_list  = []
            unet_preds_list   = []
            snr_in_list       = []
            snr_out_unet_list = []

            for i in tqdm(range(len(test_waves)),
                          desc=f"    {cond} {snr:+d}dB", leave=False):
                wav_np   = test_waves[i]
                noisy    = noisy_arr[i]
                denoised = denoised_arr[i]

                noisy_preds_list.append(infer_mfcc(noisy))
                unet_preds_list.append(infer_mfcc(denoised))
                snr_in_list.append(measure_snr_input(wav_np, noisy))
                snr_out_unet_list.append(measure_snr_output(wav_np, denoised))

            true_arr  = test_labels_audio
            acc_noisy = np.mean(np.array(noisy_preds_list) == true_arr) * 100
            acc_unet  = np.mean(np.array(unet_preds_list)  == true_arr) * 100

            multirun_records.append({
                "run"             : run_idx + 1,
                "seed"            : run_seed,
                "condition"       : cond,
                "SNR_target"      : snr,
                "acc_noisy"       : acc_noisy,
                "acc_unet"        : acc_unet,
                "unet_recovery"   : acc_unet - acc_noisy,
                "snr_in"          : np.mean(snr_in_list),
                "snr_out_unet"    : np.mean(snr_out_unet_list),
                "unet_snr_improve": np.mean(snr_out_unet_list) - np.mean(snr_in_list),
            })
            print(f"    SNR={snr:>+4}dB | noisy={acc_noisy:.2f}% | "
                  f"unet={acc_unet:.2f}% | dacc={acc_unet-acc_noisy:+.2f}%")

df_multirun = pd.DataFrame(multirun_records)
df_multirun.to_csv(OUT_RESULTS / "mfcc_unet_multirun_raw.csv", index=False)
print(f"\nSaved raw results -> {OUT_RESULTS / 'mfcc_unet_multirun_raw.csv'}")
print(f"Clean accuracy across {N_RUNS} runs: "
      f"{np.mean(clean_run_acc):.2f}% ± {np.std(clean_run_acc, ddof=1):.2f}%")



 RUN 1/5 (seed=42)
  Loading cached checkpoint: /content/drive/MyDrive/DSP Project/mfcc_unet_denoiser_multirun/models/unet_run1_seed42_traffic-gaussian.pth
  Clean accuracy this run: 55.75%

  Condition: TRAFFIC


    traffic +20dB:   0%|          | 0/400 [00:00<?, ?it/s]

    SNR= +20dB | noisy=41.00% | unet=48.25% | dacc=+7.25%


    traffic +15dB:   0%|          | 0/400 [00:00<?, ?it/s]

    SNR= +15dB | noisy=33.75% | unet=44.25% | dacc=+10.50%


...

    SNR=  +0dB | noisy=3.00% | unet=27.00% | dacc=+24.00%


    gaussian -5dB:   0%|          | 0/400 [00:00<?, ?it/s]

    SNR=  -5dB | noisy=2.25% | unet=20.75% | dacc=+18.50%


    gaussian -10dB:   0%|          | 0/400 [00:00<?, ?it/s]

    SNR= -10dB | noisy=2.00% | unet=16.50% | dacc=+14.50%

Saved raw results -> /content/drive/MyDrive/DSP Project/mfcc_unet_denoiser_multirun/results/mfcc_unet_multirun_raw.csv
Clean accuracy across 5 runs: 55.75% ± 0.00%


In [13]:
# ============================================================
# Cell 13 — Summary: mean ± std, plus paired significance tests
# ============================================================

def paired_tests(noisy_vals, unet_vals):
    """Paired t-test and Wilcoxon signed-rank test: U-Net vs noisy accuracy across runs."""
    try:
        _, t_p = scipy_stats.ttest_rel(unet_vals, noisy_vals)
    except Exception:
        t_p = np.nan
    try:
        _, w_p = scipy_stats.wilcoxon(unet_vals, noisy_vals)
    except Exception:
        w_p = np.nan
    return t_p, w_p

agg_rows = []
for cond in NOISE_GENERATORS:
    for snr in SNR_LEVELS:
        sub = df_multirun[
            (df_multirun["condition"] == cond) &
            (df_multirun["SNR_target"] == snr)
        ].sort_values("run")

        t_p, w_p = paired_tests(sub["acc_noisy"].values, sub["acc_unet"].values)

        agg_rows.append({
            "condition"          : cond,
            "SNR_target"         : snr,
            "acc_noisy_mean"     : sub["acc_noisy"].mean(),
            "acc_noisy_std"      : sub["acc_noisy"].std(ddof=1),
            "acc_unet_mean"      : sub["acc_unet"].mean(),
            "acc_unet_std"       : sub["acc_unet"].std(ddof=1),
            "recovery_mean"      : sub["unet_recovery"].mean(),
            "recovery_std"       : sub["unet_recovery"].std(ddof=1),
            "snr_in_mean"        : sub["snr_in"].mean(),
            "snr_in_std"         : sub["snr_in"].std(ddof=1),
            "snr_out_unet_mean"  : sub["snr_out_unet"].mean(),
            "snr_out_unet_std"   : sub["snr_out_unet"].std(ddof=1),
            "snr_improve_mean"   : sub["unet_snr_improve"].mean(),
            "snr_improve_std"    : sub["unet_snr_improve"].std(ddof=1),
            "t_p"                : t_p,
            "w_p"                : w_p,
        })

df_agg = pd.DataFrame(agg_rows)
df_agg.to_csv(OUT_RESULTS / "mfcc_unet_multirun_summary.csv", index=False)

clean_mean = np.mean(clean_run_acc)
clean_std  = np.std(clean_run_acc, ddof=1)

def fmt_p(p):
    if pd.isna(p):
        return " n/a "
    return "<0.001" if p < 0.001 else f"{p:.3f}"

print(f"{'='*108}")
print(f" MULTI-RUN RESULTS (N_RUNS={N_RUNS})")
print(f" Model : MFCC + Subspace Discriminant Ensemble | Denoiser: U-Net STFT mask")
print(f"{'='*108}")
print(f"\n CLEAN BASELINE: {clean_mean:.2f}% ± {clean_std:.2f}% (mean ± std across {N_RUNS} runs)\n")

COL = dict(snr=6, acc=16, rec=12, p=8)

for cond in NOISE_GENERATORS:
    title = f"{cond.capitalize()} Noise"
    print(f"\n{title}")
    hdr = (f" {'SNR':>{COL['snr']}} | {'Noisy':^{COL['acc']}} | "
           f"{'U-Net':^{COL['acc']}} | {'Recovery':^{COL['rec']}} | "
           f"{'t-p':^{COL['p']}} | {'W-p':^{COL['p']}}")
    print("-" * len(hdr))
    print(hdr)
    print("-" * len(hdr))

    sub = df_agg[df_agg["condition"] == cond]
    for _, r in sub.iterrows():
        noisy_str = f"{r.acc_noisy_mean:.2f}±{r.acc_noisy_std:.2f}"
        unet_str  = f"{r.acc_unet_mean:.2f}±{r.acc_unet_std:.2f}"
        rec_str   = f"{r.recovery_mean:+.2f}%"
        print(f" {int(r.SNR_target):>{COL['snr']-2}}dB | "
              f"{noisy_str:^{COL['acc']}} | {unet_str:^{COL['acc']}} | "
              f"{rec_str:^{COL['rec']}} | {fmt_p(r.t_p):^{COL['p']}} | "
              f"{fmt_p(r.w_p):^{COL['p']}}")

print(f"\nSaved -> {OUT_RESULTS / 'mfcc_unet_multirun_summary.csv'}")
print(f"\nNote: with N_RUNS={N_RUNS}, the Wilcoxon signed-rank test's smallest possible "
      f"two-sided p-value is 1/2^{N_RUNS-1} = {1/2**(N_RUNS-1):.4f}, so it cannot show "
      f"p < 0.05 significance even when U-Net wins every single run. The paired t-test "
      f"does not have this floor. If you need W-p to be able to cross 0.05, raise N_RUNS "
      f"to at least 6.")

 MULTI-RUN RESULTS (N_RUNS=5)
 Model : MFCC + Subspace Discriminant Ensemble | Denoiser: U-Net STFT mask

 CLEAN BASELINE: 55.75% ± 0.00% (mean ± std across 5 runs)


Traffic Noise
----------------------------------------------------------------------------------
    SNR |      Noisy       |      U-Net       |   Recovery   |   t-p    |   W-p   
----------------------------------------------------------------------------------
   20dB |    40.90±0.80    |    45.25±2.47    |    +4.35%    |  0.020   |  0.062  
   15dB |    34.20±0.62    |    39.75±3.40    |    +5.55%    |  0.023   |  0.062  
   10dB |    26.85±1.10    |    33.10±5.28    |    +6.25%    |  0.038   |  0.062  
    5dB |    19.35±0.65    |    25.80±4.86    |    +6.45%    |  0.050   |  0.062  
    0dB |    11.55±0.91    |    18.45±3.58    |    +6.90%    |  0.008   |  0.062  
   -5dB |    5.90±0.80     |    13.45±2.75    |    +7.55%    |  0.004   |  0.062  
  -10dB |    2.55±0.33     |    8.60±2.08     |    +6.05%    |  0.003   